# Explore U.S. natural gas in one request

This notebook fetches weekly Lower 48 working gas storage, U.S. population-weighted degree days and Henry Hub spot prices from the Natural Gas Data API. It checks the weekly calendar, then examines the most recent 52 reported storage weeks.

Run the cells in order in Jupyter with Python 3, pandas and matplotlib. The public endpoint requires no account or API key. Outputs are deliberately cleared. Run all cells to load the selected mirror and see its source edition. Observations can be revised; this is not a reconstruction of historical publication vintages.

In [ ]:
import json
import os
from urllib.request import urlopen

import pandas as pd
import matplotlib.pyplot as plt

default_base = (
    "https://neurax-jb.github.io/"
    "natural-gas-data-api/lab/api/"
)
# Set NATGAS_API_BASE only when using a local or hosted mirror.
base = os.getenv("NATGAS_API_BASE", default_base).rstrip("/")
with urlopen(f"{base}/v1/index.json", timeout=25) as response:
    catalog = json.load(response)
print("Source snapshot (UTC):", catalog["built_at_utc"])
with urlopen(f"{base}/v1/weekly.json", timeout=25) as response:
    weekly = pd.DataFrame(json.load(response))

for field in ["week_ending", "period_start", "period_end"]:
    weekly[field] = pd.to_datetime(weekly[field], format="%Y-%m-%d")
weekly = weekly.set_index("week_ending").sort_index()
print(f"{len(weekly):,} weekly observations | {weekly.index.min():%Y-%m-%d} to {weekly.index.max():%Y-%m-%d}")

## Check the dates and price denominator

Each Friday is a storage reference date. Heating and cooling totals sum seven days from the previous Friday through Thursday. The spot price averages only reported quotes in the same window. The `spot_days` count lets you distinguish five-price weeks, holiday weeks and weeks with no quotes.

In [ ]:
assert weekly.index.is_unique
assert (weekly.index.dayofweek == 4).all()
assert (weekly.index - weekly["period_start"] == pd.Timedelta(days=7)).all()
assert (weekly.index - weekly["period_end"] == pd.Timedelta(days=1)).all()
assert weekly["spot_days"].between(0, 7).all()
assert weekly["spot_avg_usd_mmbtu"].isna().eq(weekly["spot_days"].eq(0)).all()

columns = ["working_gas_bcf", "change_bcf", "hdd", "cdd", "spot_avg_usd_mmbtu", "spot_days"]
print(weekly[columns].tail(5).to_string())

## Compare net stock decreases and increases

A positive change is a net stock increase; a negative change is a decrease. These are differences in reported working-gas stocks. Reclassifications can make them differ from physical injections or withdrawals. Count both within the latest 52 reported weeks. Degree days measure temperature exposure, not gas demand; compare their averages by stock-change direction without assuming causation.

In [ ]:
recent = weekly.tail(52).copy()
def flow_label(change):
    if pd.isna(change):
        return "Unknown"
    if change < 0:
        return "Stock decrease"
    if change > 0:
        return "Stock increase"
    return "Unchanged"

recent["flow"] = recent["change_bcf"].map(flow_label)
summary = recent.groupby("flow").agg(
    weeks=("flow", "size"),
    mean_change_bcf=("change_bcf", "mean"),
    mean_hdd=("hdd", "mean"),
    mean_cdd=("cdd", "mean"),
    weeks_with_price=("spot_avg_usd_mmbtu", "count"),
)
print(summary.round(1).to_string())

## Plot stocks and spot prices

The panels share a time axis. Separate units let you inspect the timing of movements without imposing a relationship between the scales.

In [ ]:
import matplotlib.dates as mdates

fig, (stocks_ax, spot_ax) = plt.subplots(2, 1, sharex=True, figsize=(8.3, 4.8), constrained_layout=True)
fig.suptitle("Lower 48 stocks and Henry Hub spot price • latest 52 reported weeks", x=0.08, ha="left", fontsize=12)
for axis, field, color, units in [
    (stocks_ax, "working_gas_bcf", "#286f65", "Storage (Bcf)"),
    (spot_ax, "spot_avg_usd_mmbtu", "#9c5039", "Mean spot ($/MMBtu)"),
]:
    axis.plot(recent.index, recent[field], color=color, linewidth=2)
    axis.set_ylabel(units)
    axis.grid(axis="y", alpha=0.2)
    axis.spines[["top", "right"]].set_visible(False)
locator = mdates.AutoDateLocator(minticks=4, maxticks=7)
spot_ax.xaxis.set_major_locator(locator)
spot_ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
spot_ax.set_xlabel("EIA storage week ending Friday")
plt.show()

## Reuse the data

Use `v1/years/2024.json` for one calendar year or `v1/recent.json` for the latest 52 reported weeks. `v1/latest.json` returns a single object. To inspect the range, units and retrieval timestamp of each input, open the [data-status page](https://neurax-jb.github.io/natural-gas-data-api/lab/api/status.html). Storage totals and spot quotes come from the U.S. Energy Information Administration. Observed degree days come from NOAA's Climate Prediction Center. Archived values may incorporate revisions made after the original publication day.